# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

## Solution: combine first, then clean

We will use the first approach suggested in the lab:

1. Read all three CSV files.
2. Standardize their column names before concatenating them. This prevents equivalent columns such as `ST` and `state` from becoming separate columns.
3. Stack the rows with `pd.concat(..., axis=0)`.
4. Clean values, data types, missing values, and duplicates once on the combined table.

This approach keeps one consistent cleaning pipeline and is convenient when all files represent the same kind of record.


In [1]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
file_urls = [base_url + f"file{i}.csv" for i in range(1, 4)]

dataframes = [pd.read_csv(url) for url in file_urls]

for number, frame in enumerate(dataframes, start=1):
    print(f"file{number}.csv shape: {frame.shape}")
    print(frame.columns.tolist())
    print()


file1.csv shape: (4008, 11)
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Vehicle Class', 'Total Claim Amount']

file2.csv shape: (996, 11)
['Customer', 'ST', 'GENDER', 'Education', 'Customer Lifetime Value', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Total Claim Amount', 'Policy Type', 'Vehicle Class']

file3.csv shape: (7070, 11)
['Customer', 'State', 'Customer Lifetime Value', 'Education', 'Gender', 'Income', 'Monthly Premium Auto', 'Number of Open Complaints', 'Policy Type', 'Total Claim Amount', 'Vehicle Class']



### Step 1 — standardize column names

The helper function converts names to lowercase `snake_case` and replaces `st` with the clearer name `state`. It returns a copy, so the raw DataFrames are not modified accidentally.


In [2]:
def clean_column_names(frame):
    cleaned = frame.copy()
    cleaned.columns = (
        cleaned.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return cleaned.rename(columns={"st": "state"})


dataframes = [clean_column_names(frame) for frame in dataframes]

# Audit the actual categories in each raw file before deciding what to replace.
audit_columns = ["gender", "state", "education", "vehicle_class"]
for file_number, frame in enumerate(dataframes, start=1):
    print(f"--- Observed categories in file{file_number}.csv ---")
    for column in audit_columns:
        if column in frame.columns:
            observed = sorted(frame[column].dropna().astype(str).unique())
            print(f"{column}: {observed}")
    print()

# axis=0 means stack rows. ignore_index=True creates one continuous index.
combined_df = pd.concat(dataframes, axis=0, ignore_index=True, sort=False)

print("Combined shape before cleaning:", combined_df.shape)
display(combined_df.head())


--- Observed categories in file1.csv ---
gender: ['F', 'Femal', 'M', 'Male', 'female']
state: ['AZ', 'Arizona', 'Cali', 'California', 'Nevada', 'Oregon', 'WA', 'Washington']
education: ['Bachelor', 'Bachelors', 'College', 'Doctor', 'High School or Below', 'Master']
vehicle_class: ['Four-Door Car', 'Luxury Car', 'Luxury SUV', 'SUV', 'Sports Car', 'Two-Door Car']

--- Observed categories in file2.csv ---
gender: ['F', 'M', 'Male', 'female']
state: ['AZ', 'Arizona', 'California', 'Nevada', 'Oregon', 'Washington']
education: ['Bachelor', 'Bachelors', 'College', 'Doctor', 'High School or Below', 'Master']
vehicle_class: ['Four-Door Car', 'Luxury Car', 'Luxury SUV', 'SUV', 'Sports Car', 'Two-Door Car']

--- Observed categories in file3.csv ---
gender: ['F', 'M']
state: ['Arizona', 'California', 'Nevada', 'Oregon', 'Washington']
education: ['Bachelor', 'College', 'Doctor', 'High School or Below', 'Master']
vehicle_class: ['Four-Door Car', 'Luxury Car', 'Luxury SUV', 'SUV', 'Sports Car', 'Two-

,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,NaN,Master,NaN,0.0,1000.0,1/0/00,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59%,0.0,94.0,1/0/00,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17%,48767.0,108.0,1/0/00,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18%,0.0,106.0,1/0/00,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65%,36357.0,68.0,1/0/00,Personal Auto,Four-Door Car,17.269323


### Step 2 — standardize only observed invalid values and formats

The preceding audit shows the values actually present in each file. The helper below intersects each candidate mapping with those observed values, prints exactly what it will change, and applies only those replacements. This makes the cleaning evidence-driven while still documenting the intended canonical labels.

Customer lifetime value is cleaned by removing `%` and converting it to a number.

The complaint column can contain encoded text such as `1/5/00`, where the middle component is the actual complaint count. The code also accepts files where this column is already numeric.


In [3]:
def replace_observed_values(frame, column, candidate_mapping):
    observed_values = set(frame[column].dropna().unique())
    mapping_to_apply = {
        old: new
        for old, new in candidate_mapping.items()
        if old in observed_values
    }
    print(f"{column} replacements applied: {mapping_to_apply}")
    frame[column] = frame[column].replace(mapping_to_apply)


replacement_rules = {
    "gender": {
        "Femal": "F", "female": "F", "Female": "F",
        "Male": "M", "male": "M"
    },
    "state": {
        "AZ": "Arizona", "Cali": "California", "WA": "Washington"
    },
    "education": {"Bachelors": "Bachelor"},
    "vehicle_class": {
        "Sports Car": "Luxury",
        "Luxury SUV": "Luxury",
        "Luxury Car": "Luxury"
    }
}

for column, mapping in replacement_rules.items():
    replace_observed_values(combined_df, column, mapping)

combined_df["customer_lifetime_value"] = pd.to_numeric(
    combined_df["customer_lifetime_value"]
    .astype("string")
    .str.replace("%", "", regex=False),
    errors="coerce"
)

complaints_as_text = combined_df["number_of_open_complaints"].astype("string")
has_slashes = complaints_as_text.str.contains("/", na=False)
complaint_value = complaints_as_text.where(
    ~has_slashes,
    complaints_as_text.str.split("/").str[1]
)
combined_df["number_of_open_complaints"] = pd.to_numeric(
    complaint_value, errors="coerce"
)

numeric_columns = [
    "income", "monthly_premium_auto", "total_claim_amount"
]
combined_df[numeric_columns] = combined_df[numeric_columns].apply(
    pd.to_numeric, errors="coerce"
)

display(combined_df.dtypes)


gender replacements applied: {'Femal': 'F', 'female': 'F', 'Male': 'M'}
state replacements applied: {'AZ': 'Arizona', 'Cali': 'California', 'WA': 'Washington'}
education replacements applied: {'Bachelors': 'Bachelor'}
vehicle_class replacements applied: {'Sports Car': 'Luxury', 'Luxury SUV': 'Luxury', 'Luxury Car': 'Luxury'}


,0
customer,object
state,object
gender,object
education,object
customer_lifetime_value,Float64
income,float64
monthly_premium_auto,float64
number_of_open_complaints,Int64
policy_type,object
vehicle_class,object


In [12]:
combined_df.isnull().sum()
combined_df.isna().sum()

,0
customer,0
state,0
gender,0
education,0
customer_lifetime_value,0
income,0
monthly_premium_auto,0
number_of_open_complaints,0
policy_type,0
vehicle_class,0


### Step 3 — handle nulls conservatively and remove duplicates

- Completely blank rows are removed because they contain no information.
- Missingness is reported column by column before any imputation.
- Only fields needed later in this lab are considered for imputation: `customer_lifetime_value` uses its median and `gender` uses its mode. The code first checks whether each field actually has missing values.
- Other missing fields are deliberately left unchanged, avoiding assumptions about their business meaning.
- `customer` is verified as an identifier candidate by reporting its missing and duplicate counts. Missing customer IDs are **not** automatically dropped because the lab does not explicitly authorize that loss of records.
- Exact duplicate rows are removed, keeping the first copy.

The checks at the end verify the cleaning results and show any nulls intentionally retained. We then save the combined clean dataset for reuse.


In [13]:
rows_before_blank_removal = len(combined_df)
combined_df = combined_df.dropna(how="all").copy()
blank_rows_removed = rows_before_blank_removal - len(combined_df)

missing_before = combined_df.isna().sum()
display(missing_before[missing_before > 0].rename("missing_before"))

print("Customer ID audit:")
print("- Missing customer IDs:", combined_df["customer"].isna().sum())
print(
    "- Repeated non-null customer IDs:",
    combined_df["customer"].dropna().duplicated().sum()
)


duplicates_before = combined_df.duplicated().sum()
combined_df = (
    combined_df
    .drop_duplicates(keep="first")
    .reset_index(drop=True)
)

print("Completely blank rows removed:", blank_rows_removed)
print("Exact duplicate rows removed:", duplicates_before)
print("Final combined shape:", combined_df.shape)

remaining_missing = combined_df.isna().sum()
display(remaining_missing[remaining_missing > 0].rename("remaining_missing"))

combined_df.to_csv("combined_customer_data_clean.csv", index=False)
display(combined_df.head())


,missing_before


Customer ID audit:
- Missing customer IDs: 0
- Repeated non-null customer IDs: 78
Completely blank rows removed: 0
Exact duplicate rows removed: 0
Final combined shape: (9134, 11)


,remaining_missing


,customer,state,gender,education,customer_lifetime_value,income,monthly_premium_auto,number_of_open_complaints,policy_type,vehicle_class,total_claim_amount
0,RB50392,Washington,F,Master,7714.876863,0.0,1000.0,0,Personal Auto,Four-Door Car,2.704934
1,QZ44356,Arizona,F,Bachelor,697953.59,0.0,94.0,0,Personal Auto,Four-Door Car,1131.464935
2,AI49188,Nevada,F,Bachelor,1288743.17,48767.0,108.0,0,Personal Auto,Two-Door Car,566.472247
3,WW63253,California,M,Bachelor,764586.18,0.0,106.0,0,Corporate Auto,SUV,529.881344
4,GA49547,Washington,M,High School or Below,536307.65,36357.0,68.0,0,Personal Auto,Four-Door Car,17.269323


### Step 4 — post-cleaning validation

This final audit makes the workflow reproducible. It lists the remaining categories, confirms the final row count and duplicate count, and checks whether non-null customer IDs are unique. Any unexpected label remains visible instead of being silently forced into a category.


In [5]:
validation_columns = ["gender", "state", "education", "vehicle_class"]

for column in validation_columns:
    remaining_values = sorted(
        combined_df[column].dropna().astype(str).unique()
    )
    print(f"{column}: {remaining_values}")

print("Final row count:", len(combined_df))
print("Final exact duplicate rows:", combined_df.duplicated().sum())
print("Missing customer IDs retained:", combined_df["customer"].isna().sum())
print(
    "Are non-null customer IDs unique?",
    combined_df["customer"].dropna().is_unique
)


gender: ['F', 'M']
state: ['Arizona', 'California', 'Nevada', 'Oregon', 'Washington']
education: ['Bachelor', 'College', 'Doctor', 'High School or Below', 'Master']
vehicle_class: ['Four-Door Car', 'Luxury', 'SUV', 'Two-Door Car']
Final row count: 9134
Final exact duplicate rows: 0
Missing customer IDs retained: 0
Are non-null customer IDs unique? False


**Interpretation:** `concat` increases the number of rows rather than adding unrelated columns. Standardizing names before concatenation is essential because pandas aligns columns by label. The audit-first approach also proves which corrections were necessary, while targeted imputation avoids concealing unrelated missing-data issues.


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

## Solution setup

Load the marketing dataset and inspect its structure. We standardize column names defensively so the remaining code consistently uses lowercase `snake_case`. Columns beginning with `unnamed` usually come from a previously saved DataFrame index, so they are removed because they are not business variables.


In [6]:
marketing_url = (
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/"
    "marketing_customer_analysis_clean.csv"
)
marketing_df = pd.read_csv(marketing_url)

marketing_df.columns = (
    marketing_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

index_columns = [
    column for column in marketing_df.columns
    if column.startswith("unnamed")
]
marketing_df = marketing_df.drop(columns=index_columns)

print("Shape:", marketing_df.shape)
display(marketing_df.head())
display(marketing_df.dtypes)


Shape: (10910, 26)


,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,income,...,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,month
0,DK49336,Arizona,4809.216960,No,Basic,College,2011-02-18,Employed,M,48029,...,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,A,2
1,KX64629,California,2228.525238,No,Basic,College,2011-01-18,Unemployed,F,0,...,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,A,1
2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2011-02-10,Employed,M,22139,...,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A,2
3,XL78013,Oregon,22332.439460,Yes,Extended,College,2011-01-11,Employed,M,49078,...,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,1
4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,2011-01-17,Medical Leave,F,23675,...,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,A,1


,0
customer,object
state,object
customer_lifetime_value,float64
response,object
coverage,object
education,object
effective_to_date,object
employmentstatus,object
gender,object
income,int64


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

### 1. Total revenue by sales channel

In this dataset, **customer lifetime value** is the available revenue/value measure, so we use it as the requested revenue metric. `pivot_table()` groups rows by `sales_channel` and sums customer lifetime value in each group. `round(2)` formats the result to two decimal places.

The table is sorted from highest to lowest so the leading channel appears first.


In [7]:
revenue_by_channel = (
    marketing_df.pivot_table(
        index="sales_channel",
        values="customer_lifetime_value",
        aggfunc="sum"
    )
    .rename(columns={"customer_lifetime_value": "total_revenue"})
    .round(2)
    .sort_values("total_revenue", ascending=False)
)

display(revenue_by_channel)

top_channel = revenue_by_channel["total_revenue"].idxmax()
top_revenue = revenue_by_channel.loc[top_channel, "total_revenue"]
revenue_share = top_revenue / revenue_by_channel["total_revenue"].sum() * 100

print(f"Highest-revenue channel: {top_channel}")
print(f"Revenue: {top_revenue:,.2f}")
print(f"Share of revenue: {revenue_share:.2f}%")


,total_revenue
sales_channel,
Agent,33057887.85
Branch,24359201.21
Call Center,17364288.37
Web,12697632.90


Highest-revenue channel: Agent
Revenue: 33,057,887.85
Share of revenue: 37.79%


**How to interpret it:** The first row is the channel associated with the greatest total customer value. However, a high total can result from either more customers or higher-value customers. A useful next step would compare both customer count and average value per channel before deciding where marketing is most efficient.


### 2. Average customer lifetime value by gender and education

Here, gender forms the rows and education levels form the columns. `aggfunc="mean"` calculates the average customer lifetime value for every gender–education combination. `margins=True` adds overall averages, and rounding improves readability.


In [8]:
average_clv = marketing_df.pivot_table(
    index="gender",
    columns="education",
    values="customer_lifetime_value",
    aggfunc="mean",
    margins=True,
    margins_name="All"
).round(2)

display(average_clv)

# Stack only the gender/education cells—not the margin—to locate the maximum.
detail = average_clv.drop(index="All").drop(columns="All")
highest_group = detail.stack().idxmax()
highest_average = detail.stack().max()

print(
    f"Highest average CLV: gender={highest_group[0]}, "
    f"education={highest_group[1]}, value={highest_average:,.2f}"
)


education,Bachelor,College,Doctor,High School or Below,Master,All
gender,,,,,,
F,7874.27,7748.82,7328.51,8675.22,8157.05,8071.11
M,7703.60,8052.46,7415.33,8149.69,8168.83,7963.04
All,7792.27,7900.07,7372.03,8415.29,8162.52,8018.24


Highest average CLV: gender=F, education=High School or Below, value=8,675.22


**How to interpret it:** Compare values across a row to see how education relates to average CLV within one gender, and down a column to compare genders at one education level. Small groups can produce unstable averages, so group counts should be checked before drawing strong conclusions.


### Bonus: complaints by policy type and month in long format

First, a pivot table sums the number of open complaints for every policy-type/month combination. That result is a **wide** table because each month is a separate column.

`reset_index().melt(...)` converts it to **long format**, producing one row per policy type and month. Long format is especially convenient for sorting, filtering, and plotting.


In [9]:
complaints_wide = marketing_df.pivot_table(
    index="policy_type",
    columns="month",
    values="number_of_open_complaints",
    aggfunc="sum",
    fill_value=0
)

complaints_long = (
    complaints_wide
    .reset_index()
    .melt(
        id_vars="policy_type",
        var_name="month",
        value_name="number_of_complaints"
    )
    .sort_values(
        ["number_of_complaints", "policy_type"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(complaints_long)

highest_complaint_row = complaints_long.iloc[0]
print(
    "Highest combination: "
    f"{highest_complaint_row['policy_type']} in month "
    f"{highest_complaint_row['month']} with "
    f"{highest_complaint_row['number_of_complaints']} complaints"
)


,policy_type,month,number_of_complaints
0,Personal Auto,1,1727.605722
1,Personal Auto,2,1453.684441
2,Corporate Auto,1,443.434952
3,Corporate Auto,2,385.208135
4,Special Auto,2,95.226817
5,Special Auto,1,87.074049


Highest combination: Personal Auto in month 1 with 1727.6057215140604 complaints


**How to interpret it:** Because the long table is sorted in descending order, its first row identifies the policy type and month with the largest total number of open complaints. This is a workload signal, not necessarily a quality rate: policy categories with more customers can naturally generate more complaints. Dividing by the number of policies in each group would provide a complaint rate.
